In [10]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

In [11]:
df = pd.read_csv("../data/chembl_36_cleaned_balanced.csv")
print(df.columns)

Index(['Unnamed: 0', 'standard_value', 'standard_type', 'pchembl_value',
       'mw_freebase', 'alogp', 'hba', 'hbd', 'psa', 'rtb', 'ro3_pass',
       'num_ro5_violations', 'full_mwt', 'aromatic_rings', 'heavy_atoms',
       'qed_weighted', 'np_likeness_score', 'bei', 'sei', 'le', 'lle',
       'chirality', 'clinical_trial'],
      dtype='str')


In [12]:
df = df.drop(['Unnamed: 0'], axis=1)
df.head()

,standard_value,standard_type,pchembl_value,mw_freebase,alogp,hba,hbd,psa,rtb,ro3_pass,...,aromatic_rings,heavy_atoms,qed_weighted,np_likeness_score,bei,sei,le,lle,chirality,clinical_trial
0,10.4,IC50,7.98,524.69,4.82,8.0,3.0,108.48,10.0,N,...,3.0,37.0,0.35,-1.71,15.21,7.36,0.29,3.16,2,True
1,250.0,IC50,6.60,354.45,3.21,4.0,2.0,76.02,8.0,N,...,2.0,26.0,0.71,-1.01,18.63,8.68,0.35,3.39,-1,False
2,6.4,IC50,8.19,554.54,5.97,6.0,2.0,85.25,9.0,N,...,4.0,40.0,0.20,-1.46,14.78,9.61,0.28,2.22,-1,False
3,897.0,IC50,6.05,356.25,3.54,5.0,0.0,52.83,4.0,N,...,3.0,25.0,0.67,-2.36,16.97,11.45,0.33,2.51,-1,False
4,16.0,Ki,7.80,304.32,-1.82,7.0,5.0,142.11,3.0,N,...,1.0,20.0,0.44,-0.01,25.62,5.49,0.53,9.62,-1,False


In [25]:
#print(df.select_dtypes(exclude="number").columns)
#print(df.select_dtypes(include="number").columns)
target = df['clinical_trial']
features = df.drop(['clinical_trial'], axis=1)
features_train, features_test, target_train, target_test = train_test_split(features, target, train_size=0.8, random_state=1)
catFeaturesTrain = features_train.select_dtypes(exclude="number")
catFeaturesTest = features_test.select_dtypes(exclude="number")
numFeaturesTrain = features_train.select_dtypes(include="number")
numFeaturesTest = features_test.select_dtypes(include="number")
catFeaturesTrain = pd.get_dummies(catFeaturesTrain) #One hot encode the categorical features
catFeaturesTest = pd.get_dummies(catFeaturesTest)
#catFeaturesTrain = catFeaturesTrain.drop(['standard_type_XC50'])
#print(catFeaturesTrain.columns)
#print(catFeaturesTest.columns)
stnd = StandardScaler().set_output(transform='pandas')
stnd.fit_transform(numFeaturesTrain)
numFeaturesTrain = stnd.transform(numFeaturesTrain)
numFeaturesTest = stnd.transform(numFeaturesTest)
features_train = pd.concat([numFeaturesTrain, catFeaturesTrain], axis=1)
features_test = pd.concat([numFeaturesTest, catFeaturesTest], axis=1)
features_train = features_train.drop(['standard_type_XC50'], axis=1)

In [15]:
print(features_train.columns)
print(features_test.columns)

Index(['standard_value', 'pchembl_value', 'mw_freebase', 'alogp', 'hba', 'hbd',
       'psa', 'rtb', 'num_ro5_violations', 'full_mwt', 'aromatic_rings',
       'heavy_atoms', 'qed_weighted', 'np_likeness_score', 'bei', 'sei', 'le',
       'lle', 'chirality', 'standard_type_AC50', 'standard_type_EC50',
       'standard_type_IC50', 'standard_type_Kd', 'standard_type_Ki',
       'standard_type_Potency', 'standard_type_XC50', 'ro3_pass_N',
       'ro3_pass_Y'],
      dtype='str')
Index(['standard_value', 'pchembl_value', 'mw_freebase', 'alogp', 'hba', 'hbd',
       'psa', 'rtb', 'num_ro5_violations', 'full_mwt', 'aromatic_rings',
       'heavy_atoms', 'qed_weighted', 'np_likeness_score', 'bei', 'sei', 'le',
       'lle', 'chirality', 'standard_type_AC50', 'standard_type_EC50',
       'standard_type_IC50', 'standard_type_Kd', 'standard_type_Ki',
       'standard_type_Potency', 'ro3_pass_N', 'ro3_pass_Y'],
      dtype='str')


In [26]:
lr = LogisticRegression()
r2Values = pd.DataFrame()
r2Values['Features'] = features_train.columns
r2values = []

for feature in features_train.columns:
    featuresTrain = features_train.loc[:, feature]
    featuresTest = features_test.loc[:, feature]
    lr.fit(featuresTrain.values.reshape(-1,1), target_train.values.reshape(-1,1))
    r2values.append(lr.score(featuresTest.values.reshape(-1,1), target_test.values.reshape(-1,1)))

r2Values['Test R2'] = r2values
r2Values = r2Values.sort_values(by=['Test R2'], ascending=False)
r2Values.head(7)

/opt/miniforge3/lib/python3.11/site-packages/sklearn/utils/validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/opt/miniforge3/lib/python3.11/site-packages/sklearn/utils/validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/opt/miniforge3/lib/python3.11/site-packages/sklearn/utils/validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/opt/miniforge3/lib/python3.11/site-packages/sklearn/utils/validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_

,Features,Test R2
18,chirality,0.998842
16,le,0.597370
10,aromatic_rings,0.596791
14,bei,0.596050
21,standard_type_IC50,0.594475
9,full_mwt,0.585722
2,mw_freebase,0.584750
